# Process and save the data from Kyriakopoulos et al. (2016)

In [ ]:
import pandas as pd
import numpy as np
import utils as ut

# define data directory
datadir = "/home/staff/chao/SSEinv/Nicoya/data/"

# The EXACT data file for figure 6 in Kyriakopoulos et al. (2016)
obs_disp_CK6 = "CKfig6_data.csv"
df_ck6 = pd.read_csv(datadir + obs_disp_CK6, sep=",", skiprows=1, \
                   names=['lon', 'lat', 'vx_Car', 'vy_Car', 'vz_Car'])

obs_disp_feng = "Kyriakopoulos_novolcano.csv"    # original data from Feng et al. 2012, but with volcano sites removed
# note that the height is in m, Dt in years, the original velocity data is in mm/yr
# the disp relative to the stable Caribbean plate will be used in the inversion
# From ACOS to VENA are Campaign Sites; From BIJA to VERA are Continuous Sites; From AROL to WARN are Volcano Sites
df_feng = pd.read_csv(datadir + obs_disp_feng, sep=",", skiprows=1, \
                   names=['site', 'lat', 'lon', 'height', 'Dt', 'Days', \
                          'vy_ITRF05', 'vy_std_ITRF05', 'vx_ITRF05', 'vx_std_ITRF05', 'vz_ITRF05', 'vz_std_ITRF05', \
                          'vy_Car', 'vy_std_Car', 'vx_Car', 'vx_std_Car', 'vz_Car', 'vz_std_Car'])

# concatenate them to a combined dataframe with selected columns
df_fig6 = pd.concat([
    df_ck6[['lon', 'lat', 'vx_Car', 'vy_Car', 'vz_Car']],  # already in m/yr
    df_feng[['vx_std_Car', 'vy_std_Car', 'vz_std_Car']] / 1e3,  # convert from mm/yr to m/yr
], axis=1)

print(df_fig6.head())

# save a copy of processed data for further use
save_file_name = "CKfig6_data_final.csv"
df_fig6.to_csv(datadir + save_file_name, index=False)


In [ ]:
# The EXACT data file for figure 5 in Kyriakopoulos et al. (2016)
obs_disp_CK5 = "CKfig5_data.csv"
df_ck5 = pd.read_csv(datadir + obs_disp_CK5, sep=",", skiprows=1, \
                   names=['lon', 'lat', 'vx_Car', 'vy_Car', 'vz_Car'])

# concatenate them to a combined dataframe with selected columns
df_fig5 = pd.concat([
    df_ck5[['lon', 'lat', 'vx_Car', 'vy_Car', 'vz_Car']],  # already in m/yr
    df_feng[['vx_std_Car', 'vy_std_Car', 'vz_std_Car']] / 1e3,  # convert from mm/yr to m/yr
], axis=1)

print(df_fig5.head())

# save a copy of processed data for further use
save_file_name = "CKfig5_data_final.csv"
df_fig5.to_csv(datadir + save_file_name, index=False)

## Below is my previous way of interpreting the data processing of Figure 6 in Kyriakopoulos et al. (2016)

* Although the data is VERY close to those sent directly from C. Kyriakopoulos, but i decide to use his directly and save potential troubles.
* But to keep the record, we save the procedure here

In [ ]:
# Import GNSS data
datadir = "/home/staff/chao/SSEinv/Nicoya/data/"
# obs_disp_name = "Feng_etal_JGR_2012table1.csv" # original data from Feng et al. 2012
obs_disp_name = "Kyriakopoulos_novolcano.csv"    # same as above, but with volcano sites removed
# note that the height is in m, Dt in years, the original velocity data is in mm/yr
# the disp relative to the stable Caribbean plate will be used in the inversion
# From ACOS to VENA are Campaign Sites; From BIJA to VERA are Continuous Sites; From AROL to WARN are Volcano Sites
data = pd.read_csv(datadir + obs_disp_name, sep=",", skiprows=1, \
                   names=['site', 'lat', 'lon', 'height', 'Dt', 'Days', \
                          'vy_ITRF05', 'vy_std_ITRF05', 'vx_ITRF05', 'vx_std_ITRF05', 'vz_ITRF05', 'vz_std_ITRF05', \
                          'vy_Car', 'vy_std_Car', 'vx_Car', 'vx_std_Car', 'vz_Car', 'vz_std_Car'])
data['lon'] = -1*data['lon'] # convert to east positive, as the original data is west positive

azimuth = 45  # trench azimuth in degrees

# Get the normal and parallel components along the azimuth direction
data['v_trnorm'], data['v_trpara'] = ut.project_vector_2d_matrix(data['vx_Car'], data['vy_Car'], azimuth)

# When including the trench-parallel component, we removed 11 mm/yr of northwestward block translation
# (Figure 1) that was observed across all stations southwest of the Costa Rican volcanic chain 
# [Feng et al., 2012], since this motion does not correspond to elastic behavior along the megathrust 
# interface.
# This means, all stations except 2, ACOS, VERA
v_const_para = 11     # only remove the a constant value from trench parallel component 
mask1 = ~data['site'].isin(['ACOS', 'VERA'])   # stations north of the volcanic chain are not affected
data.loc[mask1, 'v_trpara'] = data.loc[mask1, 'v_trpara'] - v_const_para

#set these 2 stations' parallel component to 0 as well
data.loc[~mask1, 'v_trpara'] = 0

#For 3 sites, LOCA, BIJA, AGUS, the residuals are large, so don't use their parallel components
mask2 = data['site'].isin(['LOCA', 'BIJA', 'AGUS'])
data.loc[mask2, 'v_trpara'] = 0

#rotate back to N and E
data['vx_Car'], data['vy_Car'] = ut.project_vector_2d_matrix(data['v_trnorm'], data['v_trpara'], -azimuth)

# Convert mm to m, needed for inversion
cols_to_convert = ['vy_ITRF05', 'vy_std_ITRF05', 'vx_ITRF05', 'vx_std_ITRF05', 'vz_ITRF05', 'vz_std_ITRF05', \
                   'vy_Car', 'vy_std_Car', 'vx_Car', 'vx_std_Car', 'vz_Car', 'vz_std_Car']
data[cols_to_convert] = data[cols_to_convert] / 1e3  # Convert mm to m

print(data.columns.tolist())
print(data.shape)
print(data.head())

# save a copy of processed data for further use
cols_to_save = ['lon', 'lat', 'vx_Car', 'vy_Car', 'vz_Car', 'vx_std_Car', 'vy_std_Car', 'vz_std_Car']
data.to_csv(datadir + "CKfig6_data_chao.csv", columns=cols_to_save, index=False)
